# House Price Predictor - King County, USA

This notebook explores and models house sale prices in **King County, Washington** using **Multiple Linear Regression (MLR)**.

The dataset contains home sales with features such as living area, bedrooms, bathrooms, condition, waterfront access, and city location. Our goal is to understand what drives price and build a model that predicts sale price in US dollars.

## Import Libraries

We import Python libraries for data handling, visualization, and machine learning. **pandas** loads the data, **matplotlib** and **seaborn** create charts, and **scikit-learn** trains and evaluates the regression model.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_style("whitegrid")

# Project paths (notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "data.csv"
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_loader import NUMERICAL_COLUMNS, TARGET_COLUMN
from model import predict_house_price

COLUMNS_TO_DROP = ["date", "street", "statezip", "country"]
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path:    {DATA_PATH}")

## Load the Dataset

We read the CSV file into a pandas DataFrame. Each row is one house sale; each column is a feature or the sale price.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

## Data Quality Checks

Before modeling, we check for **missing values**, **duplicate rows**, and **invalid prices** (price = 0). Bad rows must be removed so they do not distort the model.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Rows with price = 0: {(df[TARGET_COLUMN] == 0).sum()}")

## Data Cleaning

We remove invalid and redundant data:
- Drop rows where **price = 0** (no real sale price)
- Drop **duplicate** rows
- Drop columns not useful for MLR: date, street, statezip, country (city already captures location)

In [ ]:
rows_before = len(df)

df = df[df[TARGET_COLUMN] != 0]
df = df.drop_duplicates()
df = df.drop(columns=COLUMNS_TO_DROP)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning:  {len(df):,}")
print(f"Columns after cleaning: {list(df.columns)}")

## Exploratory Data Analysis (EDA)

These seven plots help us **see patterns** in the data before building a model: price distribution, correlations, relationships with size/bedrooms/condition/waterfront, and average prices by city.

In [ ]:
# Plot 1: Price distribution (skewed right — many cheaper homes, few very expensive)
plt.figure(figsize=(10, 6))
sns.histplot(df[TARGET_COLUMN], bins=50, kde=True, color="steelblue")
plt.title("House Price Distribution")
plt.xlabel("Price (USD)")
plt.ylabel("Number of Houses")
plt.tight_layout()
plt.show()

# Plot 2: Correlation heatmap (numerical columns only)
numerical_df = df[[TARGET_COLUMN] + NUMERICAL_COLUMNS]
plt.figure(figsize=(14, 10))
sns.heatmap(numerical_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Feature Correlation with Price")
plt.tight_layout()
plt.show()

# Plot 3: Price vs living area
plt.figure(figsize=(10, 6))
plt.scatter(df["sqft_living"], df[TARGET_COLUMN], color="steelblue", alpha=0.4)
plt.title("Price vs Living Area (sqft)")
plt.xlabel("Living Area (sqft)")
plt.ylabel("Price (USD)")
plt.tight_layout()
plt.show()

# Plot 4: Price by bedrooms
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="bedrooms", y=TARGET_COLUMN, color="steelblue")
plt.title("Price Distribution by Number of Bedrooms")
plt.tight_layout()
plt.show()

# Plot 5: Price by condition
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="condition", y=TARGET_COLUMN, color="steelblue")
plt.title("Price Distribution by House Condition")
plt.tight_layout()
plt.show()

# Plot 6: Price by waterfront
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="waterfront", y=TARGET_COLUMN, color="steelblue")
plt.title("Waterfront vs Non-Waterfront Price Comparison")
plt.tight_layout()
plt.show()

# Plot 7: Top 10 cities by average price
city_avg = (
    df.groupby("city")[TARGET_COLUMN]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
plt.figure(figsize=(12, 6))
sns.barplot(data=city_avg, x="city", y=TARGET_COLUMN, color="steelblue")
plt.title("Top 10 Cities by Average House Price")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Feature Encoding and Train/Test Split

The **city** column is text, so we convert it to **one-hot encoded** dummy columns (`drop_first=True` avoids multicollinearity). We then split data into **features (X)** and **target (y)**, and separate **80% train / 20% test**. Finally, we **scale** features with StandardScaler (fit on train only).

In [ ]:
# One-hot encode city
city_dummies = pd.get_dummies(df["city"], prefix="city", drop_first=True)
df_model = pd.concat([df.drop(columns=["city"]), city_dummies], axis=1)

# Fill any missing numerical values with median
for col in NUMERICAL_COLUMNS:
    if col in df_model.columns and df_model[col].isnull().any():
        df_model[col] = df_model[col].fillna(df_model[col].median())

X = df_model.drop(columns=[TARGET_COLUMN])
y = df_model[TARGET_COLUMN]
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features: {len(feature_names)}")
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

## Train the MLR Model

Multiple Linear Regression learns a **weight for each feature** plus an **intercept**. It finds the best linear combination of features to predict house price.

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

print("Model trained successfully.")

## Model Evaluation Metrics

We measure performance on the **test set** (unseen houses):
- **MAE**: average dollar error — easy to interpret
- **MSE**: penalizes large errors more heavily
- **RMSE**: typical error in dollars (same unit as price)
- **R²**: fraction of price variation explained (1.0 = perfect)

We also compare **train R² vs test R²** to check for overfitting.

In [ ]:
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("TEST SET METRICS")
print(f"  MAE:  ${test_mae:,.2f}  (average dollar error)")
print(f"  MSE:  ${test_mse:,.2f}  (penalizes large errors)")
print(f"  RMSE: ${test_rmse:,.2f}  (typical error in USD)")
print(f"  R2:   {test_r2:.4f}     (variation explained)")
print()
print("OVERFITTING CHECK")
print(f"  Train R2: {train_r2:.4f}")
print(f"  Test R2:  {test_r2:.4f}")

## Feature Coefficients

Each coefficient shows how much that feature affects price (after scaling). **Positive** = increases price; **Negative** = decreases price. Larger absolute values mean stronger impact.

In [ ]:
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": model.coef_})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False).drop(
    columns=["abs_coefficient"]
)

print(f"Intercept: ${model.intercept_:,.2f}\n")
coef_df.head(20)

## Actual vs Predicted and Residual Plots

- **Actual vs Predicted**: points near the red diagonal line mean accurate predictions.
- **Residuals**: errors should scatter randomly around zero for a good model.

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_test_pred, color="steelblue", alpha=0.4)
min_p = min(y_test.min(), y_test_pred.min())
max_p = max(y_test.max(), y_test_pred.max())
plt.plot([min_p, max_p], [min_p, max_p], "r--", linewidth=2, label="Perfect predictions")
plt.title("Actual vs Predicted House Prices")
plt.xlabel("Actual Price (USD)")
plt.ylabel("Predicted Price (USD)")
plt.legend()
plt.tight_layout()
plt.show()

# Residuals
residuals = y_test - y_test_pred
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test_pred, residuals, color="steelblue", alpha=0.4)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals vs Predicted Values")
axes[0].set_xlabel("Predicted Price (USD)")
axes[0].set_ylabel("Residual (USD)")
sns.histplot(residuals, kde=True, color="steelblue", ax=axes[1])
axes[1].set_title("Residual Distribution")
plt.tight_layout()
plt.show()

## Example Predictions

We predict prices for three sample houses: a small home in Kent, a mid-range home in Bellevue, and a premium waterfront home in Seattle.

In [ ]:
examples = [
    ("Small affordable house (Kent)", {
        "sqft_living": 1200, "bedrooms": 3, "bathrooms": 1.0, "floors": 1.0,
        "waterfront": 0, "view": 0, "condition": 3, "sqft_above": 1200,
        "sqft_basement": 0, "yr_built": 1985, "yr_renovated": 0, "sqft_lot": 6000, "city": "Kent",
    }),
    ("Mid-range family house (Bellevue)", {
        "sqft_living": 2200, "bedrooms": 4, "bathrooms": 2.5, "floors": 2.0,
        "waterfront": 0, "view": 0, "condition": 4, "sqft_above": 2200,
        "sqft_basement": 0, "yr_built": 2000, "yr_renovated": 0, "sqft_lot": 9000, "city": "Bellevue",
    }),
    ("Premium waterfront house (Seattle)", {
        "sqft_living": 3500, "bedrooms": 5, "bathrooms": 3.5, "floors": 2.0,
        "waterfront": 1, "view": 4, "condition": 5, "sqft_above": 3500,
        "sqft_basement": 500, "yr_built": 2010, "yr_renovated": 2015, "sqft_lot": 12000, "city": "Seattle",
    }),
]

for title, house in examples:
    price = predict_house_price(model, scaler, feature_names, house)
    print(f"\n{title}")
    print("-" * 40)
    for k, v in house.items():
        print(f"  {k}: {v}")
    print(f"  Predicted price: ${price:,.0f}")

## Conclusion

This project built a **Multiple Linear Regression** model to predict King County house prices. Key findings:

- **Living area** (`sqft_living`, `sqft_above`) and **location** (city, especially Seattle and Bellevue) are among the strongest price drivers.
- **Waterfront** access and **view** ratings increase predicted prices.
- **Bedrooms** alone can be misleading — size and location often matter more.
- The model explains a substantial portion of price variation (see test R²), with typical errors in the tens to hundreds of thousands of dollars depending on the price range.

For production use, saved model files (`house_price_model.pkl`, `scaler.pkl`) in the `outputs/` folder can score new houses via `predict_house_price()` in `src/model.py`.